# Basic vs In-market: held-out World Cup evaluation
Use the same bundle on both pods. **Basic** is the trained Basic adapter.
Run after full training completes. No collection, fitting, package installs, or model downloads occur here.
This tests trade details at observed execution times, not trade timing or profitability.
See `docs/world_cup_evaluation.md` for Mac preparation and transfer commands.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

VARIANT = "basic"  # Change to "inmarket" on the other pod.
REPO = Path("/root/poly_world_cup_eval")
BUNDLE = Path("/root/eval_data/world_cup_eval_matches")
MODEL = Path("/workspace/models/Qwen3.6-27B")
RUN = {
    "basic": Path("/workspace/base_migration/runs/basic"),
    "inmarket": Path("/workspace/inmarket_migration/runs/inmarket_16k"),
}[VARIANT]
OUT = Path("/workspace/evaluation") / VARIANT
GPU = 0
LIMIT = 0  # Full holdout. For a pilot set 20 AND use a different OUT directory.

command = [sys.executable, str(REPO / "scripts/evaluate_world_cup.py"),
    "--bundle", str(BUNDLE), "--variant", VARIANT,
    "--run-dir", str(RUN), "--model", str(MODEL),
    "--out", str(OUT), "--gpu", str(GPU), "--limit", str(LIMIT)]

def execute(args):
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            status = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if status:
        raise RuntimeError(f"Command failed with exit code {status}; inspect output above.")

print("Python:", sys.executable)
print("Run:", RUN)
print("Output:", OUT)


## Check provenance and token lengths first
This loads the tokenizer, not the model weights. No targets are truncated.


In [ ]:
execute(command + ["--check-only"])


## Evaluate
The same cell resumes an interrupted evaluation and reuses saved predictions. Weights load once per process.


In [ ]:
execute(command + ["--resume"])


In [ ]:
summary = json.loads((OUT / "summary.json").read_text())
print(json.dumps(summary, indent=2))
with (OUT / "predictions.jsonl").open() as stream:
    first = json.loads(next(stream))
print("\nFirst prediction:")
print(json.dumps(first, indent=2))


## Compare both results
Run this cell only after both result directories are on the same machine or accessible volume.
Each directory needs `identity.json`, `predictions.jsonl`, and `summary.json`.
Use joint side/outcome accuracy with numeric error coverage. These labels are all TRADE.


In [ ]:
BASIC_RESULTS = Path("/workspace/evaluation/basic")
INMARKET_RESULTS = Path("/workspace/evaluation/inmarket")
REPORT = Path("/workspace/evaluation/comparison.json")
execute([sys.executable, str(REPO / "scripts/compare_world_cup_evaluations.py"),
    "--basic", str(BASIC_RESULTS), "--inmarket", str(INMARKET_RESULTS),
    "--out", str(REPORT)])
comparison = json.loads(REPORT.read_text())
print(json.dumps(comparison, indent=2))
